 # CTDS Synthetic Recovery Validation



 **Purpose:** Benchmark whether CTDS recovers the truth when the truth is known.

 Each section is independently runnable after setup cells, produces specific

 figures or metric tables, and accumulates results into a global `RESULTS` dict.

 ## Section 0: Imports and Global Configuration

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm
import pandas as pd
import os
import pickle
from functools import partial
from typing import Dict, Tuple, Optional, List, NamedTuple
from scipy.stats import pearsonr as scipy_pearsonr
from scipy.optimize import linear_sum_assignment
from scipy.linalg import solve_discrete_lyapunov

jax.config.update("jax_enable_x64", True)

# --- Your CTDS codebase imports ---
# Adjust these paths to match your project structure
from params import (
    ParamsCTDS, ParamsCTDSInitial, ParamsCTDSDynamics,
    ParamsCTDSEmissions, ParamsCTDSConstraints
)
from models import CTDS
from simulation_utilis import (
    generate_CTDS_Params, generate_synthetic_data,
    make_A_true, make_Q_true,
    transform_true_rec, transform_true_rec_Numpy,
    pearsonr_jax
)
from init import pca_initialize_ctds, fa_initialize_ctds
# For LDS baseline
from dynamax.linear_gaussian_ssm import LinearGaussianSSM


In [2]:
# ============================================================================
# GLOBAL CONSTANTS
# ============================================================================

# Canonical architecture
N_E = 10          # excitatory neurons
N_I = 10          # inhibitory neurons
N = N_E + N_I     # total neurons = 60
D_E = 2           # excitatory latents
D_I = 2           # inhibitory latents
D = D_E + D_I     # total latent dim = 6

# Base data volume
B_TRAIN = 80      # training trials
B_TEST = 20       # test trials
T = 100           # timesteps per trial

# EM fitting
N_EM_ITERS = 100
N_SEEDS_BASE = 5       # random inits for base experiment
N_SEEDS_SWEEP = 5      # random inits for sweeps
N_EM_ITERS_SWEEP = 75  # reduced iters for sweeps
N_DATASET_SEEDS_BASE = 3
N_DATASET_SEEDS_SWEEP = 3

# Sweep values
TRIAL_SWEEP = [10, 25, 50, 80, 150]
T_SWEEP = [25, 50, 100, 200]
SIGMA_R_SWEEP = [0.05, 0.1, 0.3, 0.5, 1.0]
SIGMA_Q_SWEEP = [0.001, 0.005, 0.01, 0.05, 0.1]
D_SWEEP = [4, 6, 8, 10]
STRUCTURE_STRENGTH_SWEEP = [0.0, 0.25, 0.5, 0.75, 1.0]

# Seeds
MASTER_KEY = jr.PRNGKey(0)
DATASET_SEEDS = [jr.PRNGKey(i) for i in range(N_DATASET_SEEDS_BASE)]

# Plotting
COLORS = {
    'ctds': '#1f77b4',
    'lds': '#ff7f0e',
    'ablated': '#2ca02c',
    'scrambled': '#d62728',
    'E': '#4393c3',
    'I': '#d6604d',
}
LINESTYLES = {
    'ctds': '-',
    'lds': '--',
    'ablated': ':',
    'scrambled': '-.',
}

# Cache directory
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Figure directory
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Global results accumulator
RESULTS = {}

plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f"Config: N={N} (E={N_E}, I={N_I}), D={D} (E={D_E}, I={D_I})")
print(f"Data: B_train={B_TRAIN}, B_test={B_TEST}, T={T}")
print(f"Fitting: {N_EM_ITERS} EM iters, {N_SEEDS_BASE} seeds (base), {N_SEEDS_SWEEP} seeds (sweep)")


Config: N=20 (E=10, I=10), D=4 (E=2, I=2)
Data: B_train=80, B_test=20, T=100
Fitting: 100 EM iters, 5 seeds (base), 5 seeds (sweep)


 ## Section 1: Synthetic Data Generation Infrastructure



 Uses existing functions from `simulation_utilis.py`. The thin wrapper

 `make_experiment_params` calls the existing generators and then overrides

 R, Q, and optionally weakens the block/sign structure.

In [3]:
def make_experiment_params(
    key,
    N_E=N_E, N_I=N_I, D_E=D_E, D_I=D_I,
    sigma_q=0.01, sigma_r=0.1,
    structure_strength=1.0,
    spectral_radius=0.95,
):
    """
    Thin wrapper around existing simulation_utilis functions.
    
    Calls generate_CTDS_Params for base structure, then overrides:
      - A via make_A_true (controlled spectral radius)
      - Q via make_Q_true (controlled condition number and scale)
      - R = sigma_r * I  (clean diagonal)
      - structure_strength interpolates C from block-diagonal to dense nonneg
    
    Returns:
        params: ParamsCTDS
        ctds_model: CTDS instance
        dynamics_mask: (D,) array of +1/-1
        metadata: dict with block indices and true constraint arrays
    """
    N = N_E + N_I
    D = D_E + D_I
    
    k1, k2, k3, k4 = jr.split(key, 4)
    
    # --- Step 1: Get base params from existing generator ---
    base_params = generate_CTDS_Params(N=N, T=100, D=D, K=2, seed=k1)
    
    cell_types = base_params.constraints.cell_types
    cell_sign = base_params.constraints.cell_sign
    cell_type_dimensions = base_params.constraints.cell_type_dimensions
    cell_type_mask = base_params.constraints.cell_type_mask
    dynamics_mask = jnp.repeat(cell_sign, cell_type_dimensions)
    
    # --- Step 2: Replace A with controlled version ---
    A = make_A_true(k2, cell_type_dimensions, cell_sign,target_cond=10.0, spectral_radius=0.95)
    
    # Optionally weaken Dale sign structure in A
    """
    if structure_strength < 1.0:
        # Add small sign violations to off-diagonal entries
        D_total = A.shape[0]
        diag_mask = jnp.eye(D_total, dtype=bool)
        noise = jr.uniform(k4, A.shape, minval=0.0, maxval=0.05)
        # For E columns (sign=+1), violations are negative entries
        # For I columns (sign=-1), violations are positive entries
        violation = noise * (-dynamics_mask[None, :])  # opposite sign
        violation = violation.at[diag_mask].set(0.0)  # keep diagonal
        A = A + (1.0 - structure_strength) * violation
    """
    
    # --- Step 3: Replace Q with controlled version ---
    #Q = make_Q_true(k3, D, target_cond=8.0, scale=sigma_q)
    Q=jr.normal(k3, (D, D))
    Q = Q.T@Q + jnp.identity(D)
    Q=Q/(jnp.max(Q)*1000)
    # --- Step 4: Replace R with clean diagonal ---
    #R = sigma_r * jnp.eye(N)
    R=jnp.array(np.diag(np.random.rand( N) + 0.1)/1000)
    
    # --- Step 5: Modify C based on structure_strength ---
    C = base_params.emissions.weights
    """
    if structure_strength < 1.0:
        # Add leakage to off-diagonal blocks
        k5, k6 = jr.split(k4)
        # E-neurons x I-latents block
        leak_EI = jr.uniform(k5, (N_E, D_I), minval=0.0, maxval=0.2)
        leak_EI = (1.0 - structure_strength) * leak_EI
        C = C.at[:N_E, D_E:].set(leak_EI)
        # I-neurons x E-latents block
        leak_IE = jr.uniform(k6, (N_I, D_E), minval=0.0, maxval=0.2)
        leak_IE = (1.0 - structure_strength) * leak_IE
        C = C.at[N_E:, :D_E].set(leak_IE)
    """
    
    # --- Step 6: Build ParamsCTDS ---
    emissions = ParamsCTDSEmissions(
        weights=C, cov=R, bias=jnp.zeros(N)
    )
    dynamics = ParamsCTDSDynamics(
        weights=A, cov=Q, dynamics_mask=dynamics_mask
    )
    initial = ParamsCTDSInitial(
        mean=jnp.zeros(D), cov=jnp.eye(D) * 0.1
    )
    constraints = ParamsCTDSConstraints(
        cell_types=cell_types,
        cell_sign=cell_sign,
        cell_type_dimensions=cell_type_dimensions,
        cell_type_mask=cell_type_mask,
    )
    params = ParamsCTDS(
        emissions=emissions,
        dynamics=dynamics,
        initial=initial,
        constraints=constraints,
        observations=None,
    )
    
    # --- Step 7: Build CTDS model ---
    ctds_model = CTDS(
        emission_dim=N,
        cell_types=cell_types,
        cell_sign=cell_sign,
        cell_type_dimensions=cell_type_dimensions,
        cell_type_mask=cell_type_mask,
        region_identity=None,
        inputs_dim=None,
        state_dim=D,
    )
    
    metadata = dict(
        N_E=N_E, N_I=N_I, D_E=D_E, D_I=D_I,
        dynamics_mask=dynamics_mask,
        cell_types=cell_types,
        cell_sign=cell_sign,
        cell_type_dimensions=cell_type_dimensions,
        cell_type_mask=cell_type_mask,
    )
    
    return base_params, ctds_model, metadata


def generate_dataset(key, params, ctds_model, B, T):
    """
    Generate B trials of length T from the CTDS model.
    
    Returns:
        y: (B, T, N) observations
        x_true: (B, T, D) true latent states
    """
    keys = jr.split(key, B)
    states_list = []
    obs_list = []
    for b in range(B):
        x_b, y_b = ctds_model.sample(params, keys[b], T)
        states_list.append(x_b)
        obs_list.append(y_b)
    
    x_true = jnp.stack(states_list, axis=0)  # (B, T, D)
    y = jnp.stack(obs_list, axis=0)          # (B, T, N)
    return y, x_true


def train_test_split(y, x_true, B_train, B_test, key):
    """
    Shuffle and split trials into train/test.
    
    Returns:
        y_train, x_train, y_test, x_test
    """
    B_total = y.shape[0]
    assert B_train + B_test <= B_total
    
    perm = jr.permutation(key, B_total)
    train_idx = perm[:B_train]
    test_idx = perm[B_train:B_train + B_test]
    
    return (
        y[train_idx], x_true[train_idx],
        y[test_idx], x_true[test_idx],
    )


# Quick sanity check
print("Section 1: Data generation infrastructure defined.")
_test_key = jr.PRNGKey(999)
_test_params, _test_model, _test_meta = make_experiment_params(_test_key)
print(f"  A shape: {_test_params.dynamics.weights.shape}")
print(f"  C shape: {_test_params.emissions.weights.shape}")
print(f"  Q shape: {_test_params.dynamics.cov.shape}")
print(f"  R shape: {_test_params.emissions.cov.shape}")
print(f"  spectral_radius(A): {jnp.max(jnp.abs(jnp.linalg.eigvals(_test_params.dynamics.weights))):.4f}")
print(f"  C zero-block norm (should be ~0): "
      f"{jnp.linalg.norm(_test_params.emissions.weights[:N_E, D_E:]):.6f}")


Section 1: Data generation infrastructure defined.
  A shape: (4, 4)
  C shape: (20, 4)
  Q shape: (4, 4)
  R shape: (20, 20)
  spectral_radius(A): 0.9500
  C zero-block norm (should be ~0): 0.000000


 ## Section 2: Model Fitting Infrastructure



 Wrappers around CTDS and LDS fitting with multi-seed initialization.

 Each wrapper returns the best model (by training LL) plus all traces.

In [4]:
def _build_ctds_init_params(y_train, ctds_model, metadata, key):
    """
    Initialize CTDS params using the PCA-based pipeline from init.py.
    Returns a ParamsCTDS ready for EM.
    """
    B, T_len, N = y_train.shape
    D = metadata['D_E'] + metadata['D_I']
    
    e_mask = jnp.array([True] * metadata['N_E'] + [False] * metadata['N_I'])
    i_mask = ~e_mask
    
    init_dict = pca_initialize_ctds(
        Y=y_train,
        e_mask=e_mask,
        i_mask=i_mask,
        D_e=metadata['D_E'],
        D_i=metadata['D_I'],
        cell_types=metadata['cell_types'],
        cell_sign=metadata['cell_sign'],
        cell_type_dimensions=metadata['cell_type_dimensions'],
        cell_type_mask=metadata['cell_type_mask'],
        key=key,
        pgd_iters=300,
    )
    
    dynamics_mask = metadata['dynamics_mask']
    
    emissions = ParamsCTDSEmissions(
        weights=init_dict['C0'],
        cov=jnp.diag(init_dict['R0']),
        bias=init_dict['d0'],
    )
    dynamics = ParamsCTDSDynamics(
        weights=init_dict['A0'],
        cov=init_dict['Q0'],
        dynamics_mask=dynamics_mask,
    )
    initial = ParamsCTDSInitial(
        mean=jnp.zeros(D),
        cov=jnp.eye(D) * 0.1,
    )
    constraints = ParamsCTDSConstraints(
        cell_types=metadata['cell_types'],
        cell_sign=metadata['cell_sign'],
        cell_type_dimensions=metadata['cell_type_dimensions'],
        cell_type_mask=metadata['cell_type_mask'],
    )
    
    return ParamsCTDS(
        emissions=emissions,
        dynamics=dynamics,
        initial=initial,
        constraints=constraints,
        observations=y_train,
    )


def fit_ctds(y_train, ctds_model, metadata, n_seeds, n_iters, master_key,
             cache_prefix=None):
    """
    Multi-seed CTDS fitting. Returns best params, LL trace, all traces.
    """
    keys = jr.split(master_key, n_seeds)
    all_traces = []
    all_params = []
    
    for s in range(n_seeds):
        # Check cache
        cache_path = None
        if cache_prefix:
            cache_path = os.path.join(CACHE_DIR, f"{cache_prefix}_ctds_seed{s}.pkl")
            if os.path.exists(cache_path):
                with open(cache_path, 'rb') as f:
                    cached = pickle.load(f)
                all_params.append(cached['params'])
                all_traces.append(cached['ll_trace'])
                print(f"    CTDS seed {s}: loaded from cache (final LL={cached['ll_trace'][-1]:.4f})")
                continue
        
        # Initialize
        #init_params = _build_ctds_init_params(y_train, ctds_model, metadata, keys[s])
        init_params=ctds_model.initialize(y_train)
        
        # Run EM
        fitted_params, ll_trace = ctds_model.fit_em(
            init_params, y_train, num_iters=n_iters
        )
        
        all_params.append(fitted_params)
        all_traces.append(np.array(ll_trace))
        print(f"    CTDS seed {s}: final LL = {ll_trace[-1]:.4f}")
        
        # Cache
        if cache_path:
            with open(cache_path, 'wb') as f:
                pickle.dump({'params': fitted_params, 'll_trace': np.array(ll_trace)}, f)
    
    # Select best
    final_lls = [tr[-1] for tr in all_traces]
    best_idx = int(np.argmax(final_lls))
    
    return {
        'params': all_params[best_idx],
        'll_trace': all_traces[best_idx],
        'all_traces': all_traces,
        'all_params': all_params,
        'best_idx': best_idx,
    }


def fit_lds(y_train, D, n_seeds, n_iters, master_key, cache_prefix=None):
    """
    Multi-seed standard LDS fitting via Dynamax. Random initialization.
    """
    B, T_len, N = y_train.shape
    keys = jr.split(master_key, n_seeds)
    all_traces = []
    all_params = []
    
    for s in range(n_seeds):
        cache_path = None
        if cache_prefix:
            cache_path = os.path.join(CACHE_DIR, f"{cache_prefix}_lds_seed{s}.pkl")
            if os.path.exists(cache_path):
                with open(cache_path, 'rb') as f:
                    cached = pickle.load(f)
                all_params.append(cached['params'])
                all_traces.append(cached['ll_trace'])
                print(f"    LDS seed {s}: loaded from cache (final LL={cached['ll_trace'][-1]:.4f})")
                continue
        
        lds = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_params, lds_props = lds.initialize(keys[s])
        
        fitted_params, ll_trace = lds.fit_em(
            lds_params, lds_props, y_train,
            num_iters=n_iters,
        )
        
        all_params.append(fitted_params)
        all_traces.append(np.array(ll_trace))
        print(f"    LDS seed {s}: final LL = {ll_trace[-1]:.4f}")
        
        if cache_path:
            with open(cache_path, 'wb') as f:
                pickle.dump({'params': fitted_params, 'll_trace': np.array(ll_trace)}, f)
    
    final_lls = [tr[-1] for tr in all_traces]
    best_idx = int(np.argmax(final_lls))
    
    return {
        'params': all_params[best_idx],
        'll_trace': all_traces[best_idx],
        'all_traces': all_traces,
        'all_params': all_params,
        'best_idx': best_idx,
    }


def fit_scrambled_ctds(y_train, ctds_model, metadata, n_seeds, n_iters, master_key,
                       cache_prefix=None):
    """
    Fit CTDS with scrambled (randomly reassigned) cell-type labels.
    Keeps the same E/I ratio but shuffles which neurons are called E vs I.
    """
    k1, k2 = jr.split(master_key)
    
    # Scramble cell_type_mask
    original_mask = metadata['cell_type_mask']
    scrambled_mask = jr.permutation(k1, original_mask)
    
    # Build new metadata with scrambled labels
    scrambled_meta = dict(metadata)
    scrambled_meta['cell_type_mask'] = scrambled_mask
    
    # Rebuild CTDS model with scrambled labels
    scrambled_model = CTDS(
        emission_dim=metadata['N_E'] + metadata['N_I'],
        cell_types=metadata['cell_types'],
        cell_sign=metadata['cell_sign'],
        cell_type_dimensions=metadata['cell_type_dimensions'],
        cell_type_mask=scrambled_mask,
        region_identity=None,
        inputs_dim=None,
        state_dim=metadata['D_E'] + metadata['D_I'],
    )
    
    return fit_ctds(y_train, scrambled_model, scrambled_meta,
                    n_seeds, n_iters, k2, cache_prefix=cache_prefix)


print("Section 2: Fitting infrastructure defined.")


Section 2: Fitting infrastructure defined.


 ## Section 3: Metric Infrastructure



 All metric functions. Each takes fitted params/posteriors and returns

 scalars or small arrays.

In [5]:
# ============================================================================
# 3.1 Observation-Level Metrics
# ============================================================================

def compute_held_out_ll(fitted_params, y_test, model, model_type='ctds'):
    """
    Held-out log-likelihood, normalized per trial per timestep.
    """
    B_test, T_len, N = y_test.shape
    total_ll = 0.0
    
    for b in range(B_test):
        if model_type == 'ctds':
            ll = model.marginal_log_prob(fitted_params, y_test[b])
        else:
            # LDS via Dynamax
            ll = model.marginal_log_prob(fitted_params, y_test[b])
        total_ll += float(ll)
    
    return total_ll / (B_test * T_len)


def compute_one_step_mse(fitted_params, y_test, model_type='ctds'):
    """
    One-step-ahead prediction MSE from Kalman filter.
    """
    B_test, T_len, N = y_test.shape
    
    if model_type == 'ctds':
        A = fitted_params.dynamics.weights
        C = fitted_params.emissions.weights
    else:
        A = fitted_params.dynamics.weights
        C = fitted_params.emissions.weights
    
    total_mse = 0.0
    count = 0
    
    for b in range(B_test):
        # Get filtered means
        if model_type == 'ctds':
            from inference import DynamaxLGSSMBackend
            posterior = DynamaxLGSSMBackend.filter(fitted_params, y_test[b])
        else:
            from dynamax.linear_gaussian_ssm import lgssm_filter
            posterior = lgssm_filter(fitted_params, y_test[b])
        
        filtered_means = posterior.filtered_means  # (T, D)
        
        # One-step predictions: ŷ_{t|t-1} = C @ A @ m_{t-1}
        # Use filtered mean at t-1 to predict observation at t
        for t in range(1, T_len):
            x_pred = A @ filtered_means[t-1]
            y_pred = C @ x_pred
            mse_t = jnp.mean((y_test[b, t] - y_pred) ** 2)
            total_mse += float(mse_t)
            count += 1
    
    return total_mse / count


def compute_obs_cov_error(fitted_params, y_test, model_type='ctds'):
    """
    Relative Frobenius error between empirical and model-implied
    observation covariance.
    
    Model-implied: Σ_model = C V_∞ C^T + R
    where V_∞ solves the discrete Lyapunov equation V = A V A^T + Q.
    """
    if model_type == 'ctds':
        A = np.array(fitted_params.dynamics.weights)
        C = np.array(fitted_params.emissions.weights)
        Q = np.array(fitted_params.dynamics.cov)
        R = np.array(fitted_params.emissions.cov)
    else:
        A = np.array(fitted_params.dynamics.weights)
        C = np.array(fitted_params.emissions.weights)
        Q = np.array(fitted_params.dynamics.cov)
        R = np.array(fitted_params.emissions.cov)
    
    # Ensure R is 2D
    if R.ndim == 1:
        R = np.diag(R)
    
    # Empirical covariance from test data
    B, T_len, N = y_test.shape
    y_flat = np.array(y_test).reshape(-1, N)
    Sigma_emp = np.cov(y_flat, rowvar=False)
    
    # Model-implied stationary covariance
    # Solve V_∞ = A V_∞ A^T + Q
    sr = np.max(np.abs(np.linalg.eigvals(A)))
    if sr >= 1.0:
        return float('inf')  # unstable model
    
    V_inf = solve_discrete_lyapunov(A, Q)
    Sigma_model = C @ V_inf @ C.T + R
    
    # Relative Frobenius error
    err = np.linalg.norm(Sigma_emp - Sigma_model, 'fro')
    norm = np.linalg.norm(Sigma_emp, 'fro')
    
    return err / (norm + 1e-12)


# ============================================================================
# 3.2 Latent Recovery Metrics
# ============================================================================

def get_smoothed_latents(fitted_params, y, model_type='ctds'):
    """
    Run Kalman smoother on all trials, return smoothed means.
    Returns: (B, T, D) array of smoothed latent means.
    """
    B, T_len, _ = y.shape
    means_list = []
    covs_list = []
    
    for b in range(B):
        if model_type == 'ctds':
            from inference import DynamaxLGSSMBackend
            smoothed_means, smoothed_covs = DynamaxLGSSMBackend.smoother(
                fitted_params, y[b]
            )
        else:
            from dynamax.linear_gaussian_ssm import lgssm_smoother
            posterior = lgssm_smoother(fitted_params, y[b])
            smoothed_means = posterior.smoothed_means
            smoothed_covs = posterior.smoothed_covariances
        
        means_list.append(smoothed_means)
        covs_list.append(smoothed_covs)
    
    return jnp.stack(means_list, axis=0), jnp.stack(covs_list, axis=0)


def align_latents_ctds(x_inferred, x_true, C_rec, C_true, A_rec, Q_rec,
                       metadata):
    """
    Align CTDS recovered latents to true latents using transform_true_rec.
    Handles permutation within cell-type blocks + per-column scaling.
    
    Returns aligned (C, A, Q) and the alignment info.
    """
    # list_of_dimensions: shape (n_regions, n_cell_types)
    # For single region: shape (1, 2)
    list_of_dims = np.array([[metadata['D_E'], metadata['D_I']]])
    
    C_aligned, A_aligned, Q_aligned = transform_true_rec_Numpy(
        C_true=np.array(C_true),
        C_rec=np.array(C_rec),
        A_rec=np.array(A_rec),
        Q_rec=np.array(Q_rec),
        list_of_dimensions=list_of_dims,
        region_identity=None,
    )
    
    return jnp.array(C_aligned), jnp.array(A_aligned), jnp.array(Q_aligned)


def align_latents_lds(x_inferred, x_true):
    """
    Align LDS latents via full linear regression: W = (X^T X)^{-1} X^T X_true.
    Concatenate all trials before computing W.
    
    Returns: x_aligned (same shape as x_inferred), W
    """
    B, T_len, D = x_inferred.shape
    X_inf = np.array(x_inferred.reshape(-1, D))
    X_true = np.array(x_true.reshape(-1, D))
    
    # Least squares: W = (X_inf^T X_inf)^{-1} X_inf^T X_true
    W = np.linalg.lstsq(X_inf, X_true, rcond=None)[0]
    
    X_aligned = X_inf @ W
    return jnp.array(X_aligned.reshape(B, T_len, D)), W


def compute_latent_r2(x_aligned, x_true):
    """
    Per-latent R² (coefficient of determination), then mean.
    
    Returns: mean_r2, per_latent_r2 array of shape (D,)
    """
    B, T_len, D = x_aligned.shape
    x_a = np.array(x_aligned.reshape(-1, D))
    x_t = np.array(x_true.reshape(-1, D))
    
    r2_per_dim = np.zeros(D)
    for d in range(D):
        ss_res = np.sum((x_a[:, d] - x_t[:, d]) ** 2)
        ss_tot = np.sum((x_t[:, d] - np.mean(x_t[:, d])) ** 2)
        r2_per_dim[d] = 1.0 - ss_res / (ss_tot + 1e-12)
    
    return float(np.mean(r2_per_dim)), r2_per_dim


def compute_latent_correlation(x_aligned, x_true):
    """
    Pearson correlation per latent dimension, then mean.
    """
    B, T_len, D = x_aligned.shape
    x_a = np.array(x_aligned.reshape(-1, D))
    x_t = np.array(x_true.reshape(-1, D))
    
    corrs = np.zeros(D)
    for d in range(D):
        corrs[d] = scipy_pearsonr(x_a[:, d], x_t[:, d])[0]
    
    return float(np.mean(corrs)), corrs


def compute_subspace_angle(x_inferred, x_true):
    """
    Mean principal angle (degrees) between column spaces of
    x_inferred and x_true (concatenated across trials).
    """
    B, T_len, D = x_inferred.shape
    X1 = np.array(x_inferred.reshape(-1, D))
    X2 = np.array(x_true.reshape(-1, D))
    
    # QR decomposition for orthonormal bases
    Q1, _ = np.linalg.qr(X1)
    Q2, _ = np.linalg.qr(X2)
    Q1 = Q1[:, :D]
    Q2 = Q2[:, :D]
    
    # Singular values of Q1^T Q2 give cosines of principal angles
    _, s, _ = np.linalg.svd(Q1.T @ Q2)
    s = np.clip(s, -1, 1)
    angles_rad = np.arccos(s)
    
    return float(np.mean(np.degrees(angles_rad)))


# ============================================================================
# 3.3 Parameter Recovery Metrics
# ============================================================================

def compute_parameter_errors(A_rec, C_rec, Q_rec, R_rec,
                             A_true, C_true, Q_true, R_true):
    """
    Relative Frobenius errors for all parameter matrices.
    """
    def rel_frob(rec, true):
        return float(np.linalg.norm(np.array(rec) - np.array(true), 'fro') /
                     (np.linalg.norm(np.array(true), 'fro') + 1e-12))
    
    # Ensure R matrices are 2D
    R_rec_2d = np.array(R_rec)
    R_true_2d = np.array(R_true)
    if R_rec_2d.ndim == 1:
        R_rec_2d = np.diag(R_rec_2d)
    if R_true_2d.ndim == 1:
        R_true_2d = np.diag(R_true_2d)
    
    return {
        'err_A': rel_frob(A_rec, A_true),
        'err_C': rel_frob(C_rec, C_true),
        'err_Q': rel_frob(Q_rec, Q_true),
        'err_R': rel_frob(R_rec_2d, R_true_2d),
    }


# ============================================================================
# 3.4 Structural Recovery Metrics
# ============================================================================

def compute_sign_accuracy_A(A_recovered, dynamics_mask):
    """
    Sign accuracy of off-diagonal entries in A.
    E columns (mask=+1): off-diag should be >= 0.
    I columns (mask=-1): off-diag should be <= 0.
    """
    A = np.array(A_recovered)
    mask = np.array(dynamics_mask)
    D = A.shape[0]
    
    e_correct, e_total = 0, 0
    i_correct, i_total = 0, 0
    
    for j in range(D):
        for i in range(D):
            if i == j:
                continue  # skip diagonal
            if mask[j] == 1:  # E column
                e_total += 1
                if A[i, j] >= 0:
                    e_correct += 1
            elif mask[j] == -1:  # I column
                i_total += 1
                if A[i, j] <= 0:
                    i_correct += 1
    
    e_acc = e_correct / max(e_total, 1)
    i_acc = i_correct / max(i_total, 1)
    overall = (e_correct + i_correct) / max(e_total + i_total, 1)
    
    return {
        'E_col_sign_acc': e_acc,
        'I_col_sign_acc': i_acc,
        'overall_sign_acc': overall,
    }


def compute_block_accuracy_C(C_recovered, N_E, D_E):
    """
    Block structure accuracy of C.
    Zero-block leakage and nonneg accuracy in active blocks.
    """
    C = np.array(C_recovered)
    N, D = C.shape
    D_I = D - D_E
    N_I = N - N_E
    
    # Zero-block leakage
    off_block_1 = C[:N_E, D_E:]  # E-neurons x I-latents (should be 0)
    off_block_2 = C[N_E:, :D_E]  # I-neurons x E-latents (should be 0)
    
    leakage = (np.linalg.norm(off_block_1, 'fro') +
               np.linalg.norm(off_block_2, 'fro'))
    leakage_rel = leakage / (np.linalg.norm(C, 'fro') + 1e-12)
    
    # Nonneg accuracy in active blocks
    active_block_E = C[:N_E, :D_E]   # should be >= 0
    active_block_I = C[N_E:, D_E:]   # should be >= 0
    
    active_entries = np.concatenate([active_block_E.ravel(), active_block_I.ravel()])
    nonneg_acc = float(np.mean(active_entries >= 0))
    
    return {
        'zero_block_leakage': float(leakage_rel),
        'nonneg_accuracy': nonneg_acc,
    }


def compute_all_metrics(fitted_result, true_params, y_test, x_true_test,
                        model, model_type, metadata):
    """
    Compute all metrics for one fitted model.
    Returns a flat dict of scalar metrics.
    """
    fitted_params = fitted_result['params']
    
    # --- Observation-level ---
    held_out_ll = compute_held_out_ll(fitted_params, y_test, model, model_type)
    obs_cov_err = compute_obs_cov_error(fitted_params, y_test, model_type)
    
    # --- Latent recovery ---
    x_smoothed, x_covs = get_smoothed_latents(fitted_params, y_test, model_type)
    
    if model_type == 'ctds':
        C_aligned, A_aligned, Q_aligned = align_latents_ctds(
            x_smoothed, x_true_test,
            fitted_params.emissions.weights,
            true_params.emissions.weights,
            fitted_params.dynamics.weights,
            fitted_params.dynamics.cov,
            metadata,
        )
        # Apply alignment to latents via the same transform
        # The alignment transforms C_rec -> C_aligned, so latents transform as x -> S^{-1} x
        # We need to extract the scaling from the C alignment
        # For now, use the aligned parameters to re-smooth (simpler: compute R2 via re-projection)
        # Simpler approach: align x_smoothed directly
        list_of_dims = np.array([[metadata['D_E'], metadata['D_I']]])
        x_aligned = x_smoothed  # placeholder — see note below
        
        # Better: use subspace angle which doesn't require alignment
        mean_r2, per_r2 = compute_latent_r2(x_smoothed, x_true_test)
        mean_corr, per_corr = compute_latent_correlation(x_smoothed, x_true_test)
        
        # Parameter errors (on aligned params)
        R_rec = fitted_params.emissions.cov
        param_errs = compute_parameter_errors(
            A_aligned, C_aligned, Q_aligned, R_rec,
            true_params.dynamics.weights, true_params.emissions.weights,
            true_params.dynamics.cov, true_params.emissions.cov,
        )
        
        # Structural metrics (on aligned A)
        sign_acc = compute_sign_accuracy_A(A_aligned, metadata['dynamics_mask'])
        block_acc = compute_block_accuracy_C(C_aligned, metadata['N_E'], metadata['D_E'])
        
    else:  # LDS
        x_aligned, W = align_latents_lds(x_smoothed, x_true_test)
        mean_r2, per_r2 = compute_latent_r2(x_aligned, x_true_test)
        mean_corr, per_corr = compute_latent_correlation(x_aligned, x_true_test)
        
        # Align LDS params via H=W
        A_lds = np.array(fitted_params.dynamics.weights)
        C_lds = np.array(fitted_params.emissions.weights)
        Q_lds = np.array(fitted_params.dynamics.cov)
        W_inv = np.linalg.inv(W)
        A_aligned = W_inv @ A_lds @ W
        C_aligned = C_lds @ W
        Q_aligned = W_inv @ Q_lds @ W_inv.T
        
        R_rec = fitted_params.emissions.cov
        param_errs = compute_parameter_errors(
            A_aligned, C_aligned, Q_aligned, R_rec,
            true_params.dynamics.weights, true_params.emissions.weights,
            true_params.dynamics.cov, true_params.emissions.cov,
        )
        
        sign_acc = compute_sign_accuracy_A(A_aligned, metadata['dynamics_mask'])
        block_acc = compute_block_accuracy_C(C_aligned, metadata['N_E'], metadata['D_E'])
    
    subspace_ang = compute_subspace_angle(x_smoothed, x_true_test)
    
    metrics = {
        'held_out_ll': held_out_ll,
        'obs_cov_err': obs_cov_err,
        'latent_r2': mean_r2,
        'latent_corr': mean_corr,
        'subspace_angle': subspace_ang,
        **param_errs,
        **sign_acc,
        **block_acc,
        'per_latent_r2': per_r2,
        'per_latent_corr': per_corr,
        'A_aligned': A_aligned,
        'C_aligned': C_aligned,
        'Q_aligned': Q_aligned,
    }
    
    return metrics


print("Section 3: Metric infrastructure defined.")
print("  Observation metrics: held_out_ll, one_step_mse, obs_cov_error")
print("  Latent metrics: latent_r2, latent_corr, subspace_angle")
print("  Parameter metrics: err_A, err_C, err_Q, err_R")
print("  Structural metrics: E/I_col_sign_acc, zero_block_leakage, nonneg_accuracy")

Section 3: Metric infrastructure defined.
  Observation metrics: held_out_ll, one_step_mse, obs_cov_error
  Latent metrics: latent_r2, latent_corr, subspace_angle
  Parameter metrics: err_A, err_C, err_Q, err_R
  Structural metrics: E/I_col_sign_acc, zero_block_leakage, nonneg_accuracy


 ## Section 4: Base Recovery Experiment



 Canonical config: N=60, D=6, B_train=80, B_test=20, T=100,

 σ_q=0.01, σ_r=0.1, structure_strength=1.0.



 Generate 10 datasets, fit CTDS / LDS / Scrambled CTDS on each.

In [8]:
# 4.1 — Data generation and fitting

base_results = {}  # indexed by (seed_idx, model_name)

for seed_idx in range(N_DATASET_SEEDS_BASE):
    print(f"\n{'='*60}")
    print(f"Dataset seed {seed_idx}/{N_DATASET_SEEDS_BASE}")
    print(f"{'='*60}")
    
    key = DATASET_SEEDS[seed_idx]
    k_params, k_data, k_split, k_fit = jr.split(key, 4)
    
    # Generate ground truth and data
    true_params, ctds_model, metadata = make_experiment_params(
        k_params, sigma_q=0.01, sigma_r=0.1, structure_strength=1.0
    )
    
    B_total = B_TRAIN + B_TEST
    y_all, x_all = generate_dataset(k_data, true_params, ctds_model, B_total, T)
    y_train, x_train, y_test, x_test = train_test_split(
        y_all, x_all, B_TRAIN, B_TEST, k_split
    )
    
    prefix = f"sec4_seed{seed_idx}"
    
    # --- Fit CTDS ---
    print(f"  Fitting CTDS ({N_SEEDS_BASE} seeds)...")
    k_ctds, k_lds, k_scr = jr.split(k_fit, 3)
    ctds_result = fit_ctds(
        y_train, ctds_model, metadata,
        N_SEEDS_BASE, N_EM_ITERS, k_ctds,
        cache_prefix=prefix,
    )
    

    # --- Fit LDS ---
    print(f"  Fitting LDS ({N_SEEDS_BASE} seeds)...")
    lds_result = fit_lds(
        y_train, D, N_SEEDS_BASE, N_EM_ITERS, k_lds,
        cache_prefix=prefix,
    )
    
    # --- Fit Scrambled CTDS ---
    print(f"  Fitting Scrambled CTDS ({N_SEEDS_BASE} seeds)...")
    scr_result = fit_scrambled_ctds(
        y_train, ctds_model, metadata,
        N_SEEDS_BASE, N_EM_ITERS, k_scr,
        cache_prefix=prefix,
    )
    
    # --- Compute metrics ---
    print("  Computing metrics...")
    
    # Create LDS model for metric computation
    lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
    

    ctds_metrics = compute_all_metrics(
        ctds_result, true_params, y_test, x_test,
        ctds_model, 'ctds', metadata
    )
    """
    lds_metrics = compute_all_metrics(
        lds_result, true_params, y_test, x_test,
        lds_model, 'lds', metadata
    )
    """
    lds_metrics = compute_all_metrics(
        ctds_result, true_params, y_test, x_test,
        ctds_model, 'ctds', metadata
    )

    scr_metrics = compute_all_metrics(
        scr_result, true_params, y_test, x_test,
        ctds_model, 'ctds', metadata  # use ctds model type for metric computation
    )

    
    base_results[(seed_idx, 'ctds')] = {
        'metrics': ctds_metrics, 'result': ctds_result,
        'true_params': true_params, 'metadata': metadata,
        'y_test': y_test, 'x_test': x_test,
    }
    base_results[(seed_idx, 'lds')] = {
        'metrics': lds_metrics, 'result': lds_result,
        'true_params': true_params, 'metadata': metadata,
    }
    base_results[(seed_idx, 'scrambled')] = {
        'metrics': scr_metrics, 'result': scr_result,
        'true_params': true_params, 'metadata': metadata,
    }
    
    print(f"  CTDS LL={ctds_metrics['held_out_ll']:.4f}, "
          f"R²={ctds_metrics['latent_r2']:.3f}, "
          f"sign_acc={ctds_metrics['overall_sign_acc']:.3f}")
    print(f"  LDS  LL={lds_metrics['held_out_ll']:.4f}, "
          f"R²={lds_metrics['latent_r2']:.3f}, "
          f"sign_acc={lds_metrics['overall_sign_acc']:.3f}")
    print(f"  SCR  LL={scr_metrics['held_out_ll']:.4f}, "
          f"R²={scr_metrics['latent_r2']:.3f}")

RESULTS['base'] = base_results



Dataset seed 0/3
  Fitting CTDS (5 seeds)...
    CTDS seed 0: loaded from cache (final LL=-10.0201)
    CTDS seed 1: loaded from cache (final LL=-10.0201)
    CTDS seed 2: loaded from cache (final LL=-10.0201)
    CTDS seed 3: loaded from cache (final LL=-10.0201)
    CTDS seed 4: loaded from cache (final LL=-10.0201)
  Fitting LDS (5 seeds)...
    LDS seed 0: loaded from cache (final LL=nan)
    LDS seed 1: loaded from cache (final LL=nan)
    LDS seed 2: loaded from cache (final LL=nan)
    LDS seed 3: loaded from cache (final LL=nan)
    LDS seed 4: loaded from cache (final LL=nan)
  Fitting Scrambled CTDS (5 seeds)...
    CTDS seed 0: loaded from cache (final LL=-10.0201)
    CTDS seed 1: loaded from cache (final LL=-10.0201)
    CTDS seed 2: loaded from cache (final LL=-10.0201)
    CTDS seed 3: loaded from cache (final LL=-10.0201)
    CTDS seed 4: loaded from cache (final LL=-10.0201)
  Computing metrics...
  CTDS LL=-10.1243, R²=-1.011, sign_acc=1.000
  LDS  LL=-10.1243, R²=-1

    LDS seed 0: final LL = nan


    LDS seed 1: final LL = nan


    LDS seed 2: final LL = nan


    LDS seed 3: final LL = nan


...

    LDS seed 1: final LL = nan


    LDS seed 2: final LL = nan


    LDS seed 3: final LL = nan


    LDS seed 4: final LL = nan
  Fitting Scrambled CTDS (5 seeds)...
    CTDS seed 0: loaded from cache (final LL=-9.4652)
    CTDS seed 1: loaded from cache (final LL=-9.4652)
    CTDS seed 2: loaded from cache (final LL=-9.4652)
    CTDS seed 3: loaded from cache (final LL=-9.4652)
    CTDS seed 4: loaded from cache (final LL=-9.4652)
  Computing metrics...


LinAlgError: Array must not contain infs or NaNs

In [9]:
# 4.2 — Metric Summary Table

metric_keys = [
    'held_out_ll', 'obs_cov_err', 'latent_r2',
    'E_col_sign_acc', 'I_col_sign_acc', 'zero_block_leakage',
    'err_A', 'err_C',
]
model_names = ['ctds', 'lds', 'scrambled']
display_names = {'ctds': 'CTDS', 'lds': 'LDS', 'scrambled': 'Scrambled CTDS'}

rows = []
for model_name in model_names:
    row = {'Model': display_names[model_name]}
    for mk in metric_keys:
        vals = [base_results[(s, model_name)]['metrics'][mk]
                for s in range(N_DATASET_SEEDS_BASE)]
        row[mk] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}"
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df = summary_df.set_index('Model')
print("\n=== Base Experiment Metric Summary ===")
print(summary_df.to_string())


KeyError: (2, 'ctds')

In [ ]:
# 4.3 — Figure: True vs. Recovered Parameter Heatmaps (Thesis Figure 4.1)

# Pick representative seed (best CTDS training LL)
best_seed = max(range(N_DATASET_SEEDS_BASE),
                key=lambda s: base_results[(s, 'ctds')]['result']['ll_trace'][-1])

rep = base_results[(best_seed, 'ctds')]
true_params_rep = rep['true_params']
meta_rep = rep['metadata']

A_true = np.array(true_params_rep.dynamics.weights)
C_true = np.array(true_params_rep.emissions.weights)
Q_true = np.array(true_params_rep.dynamics.cov)
R_true = np.array(true_params_rep.emissions.cov)

A_ctds = np.array(rep['metrics']['A_aligned'])
C_ctds = np.array(rep['metrics']['C_aligned'])
Q_ctds = np.array(rep['metrics']['Q_aligned'])
R_ctds = np.array(base_results[(best_seed, 'ctds')]['result']['params'].emissions.cov)

A_lds = np.array(base_results[(best_seed, 'lds')]['metrics']['A_aligned'])
C_lds = np.array(base_results[(best_seed, 'lds')]['metrics']['C_aligned'])
Q_lds = np.array(base_results[(best_seed, 'lds')]['metrics']['Q_aligned'])
R_lds = np.array(base_results[(best_seed, 'lds')]['result']['params'].emissions.cov)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
row_labels = ['True', 'CTDS Recovered', 'LDS Recovered']
col_labels = ['A (Dynamics)', 'C (Emissions)', 'Q (Process Noise)', 'R (Obs Noise)']

# A matrices
A_vmax = max(np.abs(A_true).max(), np.abs(A_ctds).max())
for row_idx, (A_mat, label) in enumerate([(A_true, 'True'), (A_ctds, 'CTDS'), (A_lds, 'LDS')]):
    ax = axes[row_idx, 0]
    vmax_use = A_vmax if row_idx < 2 else np.abs(A_mat).max()
    norm = TwoSlopeNorm(vmin=-vmax_use, vcenter=0, vmax=vmax_use)
    im = ax.imshow(A_mat, cmap='RdBu_r', norm=norm, aspect='auto')
    # Block boundary lines
    ax.axhline(D_E - 0.5, color='k', linewidth=1.5)
    ax.axvline(D_E - 0.5, color='k', linewidth=1.5)
    # Quadrant labels
    for qi, (qr, qc, ql) in enumerate([
        (D_E//2, D_E//2, 'EE'), (D_E//2, D_E + D_I//2, 'EI'),
        (D_E + D_I//2, D_E//2, 'IE'), (D_E + D_I//2, D_E + D_I//2, 'II')
    ]):
        ax.text(qc, qr, ql, ha='center', va='center', fontsize=7,
                color='gray', fontweight='bold', alpha=0.5)
    ax.set_ylabel(label, fontsize=10, fontweight='bold')
    if row_idx == 0:
        ax.set_title(col_labels[0])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# C matrices
C_vmax = max(np.abs(C_true).max(), np.abs(C_ctds).max())
for row_idx, (C_mat, label) in enumerate([(C_true, 'True'), (C_ctds, 'CTDS'), (C_lds, 'LDS')]):
    ax = axes[row_idx, 1]
    vmax_use = C_vmax if row_idx < 2 else np.abs(C_mat).max()
    im = ax.imshow(C_mat, cmap='viridis', vmin=0, vmax=vmax_use, aspect='auto')
    # Block boundaries
    ax.axhline(N_E - 0.5, color='k', linewidth=1.5)
    ax.axvline(D_E - 0.5, color='k', linewidth=1.5)
    # Hatch zero-block regions
    for rect_args in [
        (D_E - 0.5, -0.5, D_I, N_E),    # E-neurons x I-latents
        (-0.5, N_E - 0.5, D_E, N_I),     # I-neurons x E-latents
    ]:
        rect = Rectangle((rect_args[0], rect_args[1]),
                         rect_args[2], rect_args[3],
                         linewidth=0, edgecolor='none',
                         facecolor='gray', alpha=0.15, hatch='///')
        ax.add_patch(rect)
    if row_idx == 0:
        ax.set_title(col_labels[1])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Q matrices
Q_vmax = max(np.abs(Q_true).max(), np.abs(Q_ctds).max())
for row_idx, (Q_mat, label) in enumerate([(Q_true, 'True'), (Q_ctds, 'CTDS'), (Q_lds, 'LDS')]):
    ax = axes[row_idx, 2]
    im = ax.imshow(Q_mat, cmap='coolwarm', aspect='auto',
                   vmin=-Q_vmax, vmax=Q_vmax)
    if row_idx == 0:
        ax.set_title(col_labels[2])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# R diagonal values (bar plots)
if R_true.ndim == 2:
    R_true_diag = np.diag(R_true)
else:
    R_true_diag = R_true
if R_ctds.ndim == 2:
    R_ctds_diag = np.diag(R_ctds)
else:
    R_ctds_diag = R_ctds
if R_lds.ndim == 2:
    R_lds_diag = np.diag(R_lds)
else:
    R_lds_diag = R_lds

for row_idx, (R_diag, label) in enumerate([
    (R_true_diag, 'True'), (R_ctds_diag, 'CTDS'), (R_lds_diag, 'LDS')
]):
    ax = axes[row_idx, 3]
    colors = [COLORS['E']] * N_E + [COLORS['I']] * N_I
    ax.bar(range(N), R_diag, color=colors, width=1.0, edgecolor='none')
    ax.axvline(N_E - 0.5, color='k', linewidth=1.0, linestyle='--')
    ax.set_xlim(-0.5, N - 0.5)
    if row_idx == 0:
        ax.set_title(col_labels[3])
    if row_idx == 2:
        ax.set_xlabel('Neuron index')

fig.suptitle(f'True vs. Recovered Parameters (seed {best_seed})', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_1_parameter_heatmaps.pdf'))
plt.show()


In [ ]:
# 4.4 — Figure: EM Log-Likelihood Traces (Supplementary)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))

ctds_traces = base_results[(best_seed, 'ctds')]['result']['all_traces']
lds_traces = base_results[(best_seed, 'lds')]['result']['all_traces']
ctds_best = base_results[(best_seed, 'ctds')]['result']['best_idx']
lds_best = base_results[(best_seed, 'lds')]['result']['best_idx']

for s, tr in enumerate(ctds_traces):
    color = COLORS['ctds'] if s == ctds_best else 'gray'
    lw = 2.0 if s == ctds_best else 0.5
    alpha = 1.0 if s == ctds_best else 0.4
    label = 'CTDS (best)' if s == ctds_best else ('CTDS seeds' if s == 0 else None)
    ax.plot(tr, color=color, linewidth=lw, alpha=alpha, label=label)

for s, tr in enumerate(lds_traces):
    color = COLORS['lds'] if s == lds_best else '#ffcc99'
    lw = 2.0 if s == lds_best else 0.5
    alpha = 1.0 if s == lds_best else 0.4
    ls = '--' if s == lds_best else ':'
    label = 'LDS (best)' if s == lds_best else ('LDS seeds' if s == 0 else None)
    ax.plot(tr, color=color, linewidth=lw, alpha=alpha, linestyle=ls, label=label)

ax.set_xlabel('EM Iteration')
ax.set_ylabel('Training LL / (B × T)')
ax.set_title('EM Convergence: Multi-Seed Traces')
ax.legend(loc='lower right')
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_em_traces.pdf'))
plt.show()


In [ ]:
# 4.5 — Figure: Initialization Sensitivity (NEW, Supplementary)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

for model_name, color, marker in [
    ('ctds', COLORS['ctds'], 'o'), ('lds', COLORS['lds'], 's')
]:
    traces = base_results[(best_seed, model_name)]['result']['all_traces']
    best_idx = base_results[(best_seed, model_name)]['result']['best_idx']
    
    init_lls = [tr[0] for tr in traces]  # LL after 1st iteration
    final_lls = [tr[-1] for tr in traces]
    
    ax.scatter(init_lls, final_lls, c=color, marker=marker,
               s=50, alpha=0.7, label=model_name.upper(), zorder=3)
    # Highlight best
    ax.scatter(init_lls[best_idx], final_lls[best_idx],
               c=color, marker=marker, s=150, edgecolors='black',
               linewidths=2, zorder=4)

ax.set_xlabel('LL after iteration 1 (init quality)')
ax.set_ylabel('Final LL')
ax.set_title('Initialization Sensitivity')
ax.legend()
ax.plot([ax.get_xlim()[0], ax.get_xlim()[1]],
        [ax.get_xlim()[0], ax.get_xlim()[1]],
        'k--', alpha=0.3, linewidth=0.5)
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_init_sensitivity.pdf'))
plt.show()


In [ ]:
# 4.6 — Figure: Latent Trajectory Overlays (Thesis Figure 4.4)

rep_data = base_results[(best_seed, 'ctds')]
x_test_rep = np.array(rep_data['x_test'])

# Get smoothed latents for CTDS and LDS
x_ctds_smooth, x_ctds_covs = get_smoothed_latents(
    rep_data['result']['params'], rep_data['y_test'], 'ctds'
)
x_lds_smooth, _ = get_smoothed_latents(
    base_results[(best_seed, 'lds')]['result']['params'],
    rep_data['y_test'], 'lds'
)

# Align LDS latents
x_lds_aligned, _ = align_latents_lds(x_lds_smooth, jnp.array(x_test_rep))

# Pick trial with median CTDS R²
trial_r2s = []
for b in range(x_test_rep.shape[0]):
    _, pr2 = compute_latent_r2(x_ctds_smooth[b:b+1], jnp.array(x_test_rep[b:b+1]))
    trial_r2s.append(np.mean(pr2))
median_trial = int(np.argsort(trial_r2s)[len(trial_r2s) // 2])

fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
t_axis = np.arange(T)

for d in range(D):
    row = 0 if d < D_E else 1
    col = d if d < D_E else d - D_E
    ax = axes[row, col]
    
    # True latent
    ax.plot(t_axis, x_test_rep[median_trial, :, d], 'k-',
            linewidth=2.0, label='True', zorder=3)
    
    # CTDS smoothed
    ctds_mean = np.array(x_ctds_smooth[median_trial, :, d])
    ax.plot(t_axis, ctds_mean, color=COLORS['ctds'],
            linewidth=1.5, label='CTDS')
    
    # ±1 std ribbon for CTDS
    ctds_std = np.sqrt(np.array(x_ctds_covs[median_trial, :, d, d]))
    ax.fill_between(t_axis, ctds_mean - ctds_std, ctds_mean + ctds_std,
                     color=COLORS['ctds'], alpha=0.15)
    
    # LDS aligned
    ax.plot(t_axis, np.array(x_lds_aligned[median_trial, :, d]),
            color=COLORS['lds'], linewidth=1.5, linestyle='--', label='LDS')
    
    # Per-latent R²
    _, ctds_r2 = compute_latent_r2(
        x_ctds_smooth[median_trial:median_trial+1, :, d:d+1],
        jnp.array(x_test_rep[median_trial:median_trial+1, :, d:d+1])
    )
    _, lds_r2 = compute_latent_r2(
        x_lds_aligned[median_trial:median_trial+1, :, d:d+1],
        jnp.array(x_test_rep[median_trial:median_trial+1, :, d:d+1])
    )
    
    cell_label = 'E' if d < D_E else 'I'
    ax.set_title(f'{cell_label} latent {col}  |  CTDS R²={ctds_r2[0]:.2f}  LDS R²={lds_r2[0]:.2f}',
                 fontsize=9)
    
    # Colored spine strip
    spine_color = COLORS['E'] if d < D_E else COLORS['I']
    ax.spines['left'].set_color(spine_color)
    ax.spines['left'].set_linewidth(3)
    
    if row == 1:
        ax.set_xlabel('Time')
    if col == 0:
        ax.set_ylabel('Latent value')

axes[0, 0].legend(loc='upper right', fontsize=7)
fig.suptitle(f'Latent Trajectory Recovery (trial {median_trial}, seed {best_seed})', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_4_latent_trajectories.pdf'))
plt.show()


In [ ]:
# 4.7 — Figure: Observation Covariance Scatter (Thesis Figure 4.5)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for panel_idx, (model_name, ax_title) in enumerate([('ctds', 'CTDS'), ('lds', 'LDS')]):
    ax = axes[panel_idx]
    
    result = base_results[(best_seed, model_name)]
    fitted_params = result['result']['params']
    y_test = np.array(rep_data['y_test'])
    
    # Empirical covariance
    y_flat = y_test.reshape(-1, N)
    Sigma_emp = np.cov(y_flat, rowvar=False)
    
    # Model-implied covariance
    A = np.array(fitted_params.dynamics.weights)
    C = np.array(fitted_params.emissions.weights)
    Q = np.array(fitted_params.dynamics.cov)
    R = np.array(fitted_params.emissions.cov)
    if R.ndim == 1:
        R = np.diag(R)
    
    V_inf = solve_discrete_lyapunov(A, Q)
    Sigma_model = C @ V_inf @ C.T + R
    
    # Extract upper triangle entries
    triu_idx = np.triu_indices(N)
    emp_entries = Sigma_emp[triu_idx]
    mod_entries = Sigma_model[triu_idx]
    
    # Color by block type
    colors_scatter = []
    for i, j in zip(triu_idx[0], triu_idx[1]):
        if i < N_E and j < N_E:
            colors_scatter.append(COLORS['E'])   # E-E
        elif i >= N_E and j >= N_E:
            colors_scatter.append(COLORS['I'])   # I-I
        else:
            colors_scatter.append('gray')         # E-I cross
    
    ax.scatter(emp_entries, mod_entries, c=colors_scatter, s=8, alpha=0.5)
    
    # y=x line
    lims = [min(emp_entries.min(), mod_entries.min()),
            max(emp_entries.max(), mod_entries.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5)
    
    # Pearson r and Frobenius error
    r_val = scipy_pearsonr(emp_entries, mod_entries)[0]
    frob_err = np.linalg.norm(Sigma_emp - Sigma_model, 'fro') / np.linalg.norm(Sigma_emp, 'fro')
    
    ax.text(0.05, 0.92, f'r = {r_val:.4f}\nFrob err = {frob_err:.4f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Empirical cov entry')
    ax.set_ylabel('Model-implied cov entry')
    ax.set_title(ax_title)
    ax.set_aspect('equal')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['E'], markersize=8, label='E-E'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['I'], markersize=8, label='I-I'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='E-I'),
]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8)

fig.suptitle('Observation Covariance: Empirical vs. Model-Implied', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_5_obs_covariance_scatter.pdf'))
plt.show()


 ### 4.8 Base Experiment Summary



 Write 3–5 sentences after inspecting the table and figures above.

 Template:

 - CTDS achieves held-out LL of ___ vs LDS ___ (comparable / better).

 - CTDS latent R² = ___ vs LDS ___, confirming better trajectory recovery.

 - Sign accuracy: CTDS ___ vs LDS ___ (near chance 0.5), zero-block leakage ___ vs ___.

 - Scrambled CTDS performs ___ (worse / comparable), confirming correct labeling matters.

 ## Section 5: Sweep — Number of Trials

In [ ]:
sweep_trial_results = {}

for B_train_val in TRIAL_SWEEP:
    print(f"\n--- Trial sweep: B_train = {B_train_val} ---")
    
    for seed_idx in range(N_DATASET_SEEDS_SWEEP):
        key = DATASET_SEEDS[seed_idx]
        k_params, k_data, k_split, k_fit = jr.split(key, 4)
        
        true_params, ctds_model, metadata = make_experiment_params(
            k_params, sigma_q=0.01, sigma_r=0.1
        )
        
        B_total = max(B_train_val + B_TEST, 170)  # ensure enough trials
        y_all, x_all = generate_dataset(k_data, true_params, ctds_model, B_total, T)
        y_train, x_train, y_test, x_test = train_test_split(
            y_all, x_all, B_train_val, B_TEST, k_split
        )
        
        prefix = f"sec5_B{B_train_val}_seed{seed_idx}"
        k_c, k_l = jr.split(k_fit)
        
        ctds_res = fit_ctds(y_train, ctds_model, metadata,
                            N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
        lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_res = fit_lds(y_train, D, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
        
        ctds_m = compute_all_metrics(ctds_res, true_params, y_test, x_test,
                                     ctds_model, 'ctds', metadata)
        lds_m = compute_all_metrics(lds_res, true_params, y_test, x_test,
                                    lds_model, 'lds', metadata)
        
        sweep_trial_results[(B_train_val, seed_idx, 'ctds')] = ctds_m
        sweep_trial_results[(B_train_val, seed_idx, 'lds')] = lds_m

RESULTS['trial_sweep'] = sweep_trial_results


In [ ]:
# 5.2 — Figure: Recovery vs. Number of Trials (Thesis Figure 4.2)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)
panel_metrics = [
    ('held_out_ll', 'Held-out LL / timestep'),
    ('latent_r2', 'Mean Latent R²'),
    ('err_A', 'err_A (relative Frobenius)'),
    ('E_col_sign_acc', 'E-column Sign Accuracy'),
]

for pidx, (mkey, ylabel) in enumerate(panel_metrics):
    ax = axes[pidx // 2, pidx % 2]
    
    for model_name, color, ls in [
        ('ctds', COLORS['ctds'], '-'), ('lds', COLORS['lds'], '--')
    ]:
        means, stds = [], []
        for B_val in TRIAL_SWEEP:
            vals = [sweep_trial_results[(B_val, s, model_name)][mkey]
                    for s in range(N_DATASET_SEEDS_SWEEP)]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        
        ax.plot(TRIAL_SWEEP, means, color=color, linestyle=ls,
                marker='o', markersize=4, label=model_name.upper())
        ax.fill_between(TRIAL_SWEEP, means - stds, means + stds,
                         color=color, alpha=0.15)
    
    # Reference line at base experiment (B=80)
    base_vals_ctds = [base_results[(s, 'ctds')]['metrics'][mkey]
                      for s in range(N_DATASET_SEEDS_BASE)]
    ax.axhline(np.mean(base_vals_ctds), color='gray', linestyle=':', alpha=0.5)
    
    # Star at B=80
    if B_TRAIN in TRIAL_SWEEP:
        idx_80 = TRIAL_SWEEP.index(B_TRAIN)
        ax.plot(B_TRAIN, means[idx_80], '*', color=COLORS['ctds'],
                markersize=12, zorder=5)
    
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

axes[1, 0].set_xlabel('Number of Training Trials')
axes[1, 1].set_xlabel('Number of Training Trials')
fig.suptitle('Recovery vs. Number of Trials', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_2_trial_sweep.pdf'))
plt.show()


 ## Section 6: Sweep — Sequence Length T

In [ ]:
sweep_T_results = {}

for T_val in T_SWEEP:
    print(f"\n--- T sweep: T = {T_val} ---")
    
    for seed_idx in range(N_DATASET_SEEDS_SWEEP):
        key = DATASET_SEEDS[seed_idx]
        k_params, k_data, k_split, k_fit = jr.split(key, 4)
        
        true_params, ctds_model, metadata = make_experiment_params(
            k_params, sigma_q=0.01, sigma_r=0.1
        )
        
        y_all, x_all = generate_dataset(k_data, true_params, ctds_model,
                                         B_TRAIN + B_TEST, T_val)
        y_train, x_train, y_test, x_test = train_test_split(
            y_all, x_all, B_TRAIN, B_TEST, k_split
        )
        
        prefix = f"sec6_T{T_val}_seed{seed_idx}"
        k_c, k_l = jr.split(k_fit)
        
        ctds_res = fit_ctds(y_train, ctds_model, metadata,
                            N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
        lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_res = fit_lds(y_train, D, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
        
        ctds_m = compute_all_metrics(ctds_res, true_params, y_test, x_test,
                                     ctds_model, 'ctds', metadata)
        lds_m = compute_all_metrics(lds_res, true_params, y_test, x_test,
                                    lds_model, 'lds', metadata)
        
        sweep_T_results[(T_val, seed_idx, 'ctds')] = ctds_m
        sweep_T_results[(T_val, seed_idx, 'lds')] = lds_m

RESULTS['T_sweep'] = sweep_T_results


In [ ]:
# 6.2 — Figure: T Sweep (Supplementary)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

for pidx, (mkey, ylabel) in enumerate(panel_metrics):
    ax = axes[pidx // 2, pidx % 2]
    
    for model_name, color, ls in [
        ('ctds', COLORS['ctds'], '-'), ('lds', COLORS['lds'], '--')
    ]:
        means, stds = [], []
        for T_val in T_SWEEP:
            vals = [sweep_T_results[(T_val, s, model_name)][mkey]
                    for s in range(N_DATASET_SEEDS_SWEEP)]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        
        ax.plot(T_SWEEP, means, color=color, linestyle=ls,
                marker='o', markersize=4, label=model_name.upper())
        ax.fill_between(T_SWEEP, means - stds, means + stds,
                         color=color, alpha=0.15)
    
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    
    # Secondary x-axis: total timesteps
    if pidx // 2 == 0:
        ax2 = ax.twiny()
        ax2.set_xlim([t * B_TRAIN for t in [T_SWEEP[0], T_SWEEP[-1]]])
        ax2.set_xlabel('Total timesteps (T × B_train)', fontsize=8)

axes[1, 0].set_xlabel('Sequence Length T')
axes[1, 1].set_xlabel('Sequence Length T')
fig.suptitle('Recovery vs. Sequence Length', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_T_sweep.pdf'))
plt.show()


 ## Section 7: Sweep — Observation Noise (σ_R)

In [ ]:
sweep_sigR_results = {}

for sig_r in SIGMA_R_SWEEP:
    print(f"\n--- σ_R sweep: σ_R = {sig_r} ---")
    
    for seed_idx in range(N_DATASET_SEEDS_SWEEP):
        key = DATASET_SEEDS[seed_idx]
        k_params, k_data, k_split, k_fit = jr.split(key, 4)
        
        true_params, ctds_model, metadata = make_experiment_params(
            k_params, sigma_q=0.01, sigma_r=sig_r
        )
        
        y_all, x_all = generate_dataset(k_data, true_params, ctds_model,
                                         B_TRAIN + B_TEST, T)
        y_train, x_train, y_test, x_test = train_test_split(
            y_all, x_all, B_TRAIN, B_TEST, k_split
        )
        
        # Compute SNR
        A_np = np.array(true_params.dynamics.weights)
        C_np = np.array(true_params.emissions.weights)
        Q_np = np.array(true_params.dynamics.cov)
        R_np = np.array(true_params.emissions.cov)
        if R_np.ndim == 1:
            R_np = np.diag(R_np)
        V_inf = solve_discrete_lyapunov(A_np, Q_np)
        snr = np.linalg.norm(C_np @ V_inf @ C_np.T, 'fro') / np.linalg.norm(R_np, 'fro')
        
        prefix = f"sec7_sigR{sig_r}_seed{seed_idx}"
        k_c, k_l = jr.split(k_fit)
        
        ctds_res = fit_ctds(y_train, ctds_model, metadata,
                            N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
        lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_res = fit_lds(y_train, D, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
        
        ctds_m = compute_all_metrics(ctds_res, true_params, y_test, x_test,
                                     ctds_model, 'ctds', metadata)
        lds_m = compute_all_metrics(lds_res, true_params, y_test, x_test,
                                    lds_model, 'lds', metadata)
        
        ctds_m['snr'] = snr
        lds_m['snr'] = snr
        
        sweep_sigR_results[(sig_r, seed_idx, 'ctds')] = ctds_m
        sweep_sigR_results[(sig_r, seed_idx, 'lds')] = lds_m

RESULTS['sigR_sweep'] = sweep_sigR_results


In [ ]:
# 7.2 — Figure: σ_R Sweep (Supplementary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
noise_metrics = [
    ('held_out_ll', 'Held-out LL'),
    ('latent_r2', 'Mean Latent R²'),
    ('E_col_sign_acc', 'E-col Sign Accuracy'),
]

for pidx, (mkey, ylabel) in enumerate(noise_metrics):
    ax = axes[pidx]
    
    for model_name, color, ls in [
        ('ctds', COLORS['ctds'], '-'), ('lds', COLORS['lds'], '--')
    ]:
        means, stds = [], []
        for sig_r in SIGMA_R_SWEEP:
            vals = [sweep_sigR_results[(sig_r, s, model_name)][mkey]
                    for s in range(N_DATASET_SEEDS_SWEEP)]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        
        ax.plot(SIGMA_R_SWEEP, means, color=color, linestyle=ls,
                marker='o', markersize=4, label=model_name.upper())
        ax.fill_between(SIGMA_R_SWEEP, means - stds, means + stds,
                         color=color, alpha=0.15)
    
    ax.set_xscale('log')
    ax.set_xlabel('σ_R')
    ax.set_ylabel(ylabel)
    ax.axvline(0.1, color='gray', linestyle=':', alpha=0.5, label='canonical')
    if pidx == 2:
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3, label='chance')
    ax.legend(fontsize=7)

fig.suptitle('Recovery vs. Observation Noise σ_R', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_sigR_sweep.pdf'))
plt.show()


 ## Section 8: Sweep — Dynamics Noise (σ_Q)

In [ ]:
sweep_sigQ_results = {}

for sig_q in SIGMA_Q_SWEEP:
    print(f"\n--- σ_Q sweep: σ_Q = {sig_q} ---")
    
    for seed_idx in range(N_DATASET_SEEDS_SWEEP):
        key = DATASET_SEEDS[seed_idx]
        k_params, k_data, k_split, k_fit = jr.split(key, 4)
        
        true_params, ctds_model, metadata = make_experiment_params(
            k_params, sigma_q=sig_q, sigma_r=0.1
        )
        
        y_all, x_all = generate_dataset(k_data, true_params, ctds_model,
                                         B_TRAIN + B_TEST, T)
        y_train, x_train, y_test, x_test = train_test_split(
            y_all, x_all, B_TRAIN, B_TEST, k_split
        )
        
        prefix = f"sec8_sigQ{sig_q}_seed{seed_idx}"
        k_c, k_l = jr.split(k_fit)
        
        ctds_res = fit_ctds(y_train, ctds_model, metadata,
                            N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
        lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_res = fit_lds(y_train, D, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
        
        ctds_m = compute_all_metrics(ctds_res, true_params, y_test, x_test,
                                     ctds_model, 'ctds', metadata)
        lds_m = compute_all_metrics(lds_res, true_params, y_test, x_test,
                                    lds_model, 'lds', metadata)
        
        # Also record spectral radius of recovered A
        ctds_m['spec_radius_A'] = float(jnp.max(jnp.abs(
            jnp.linalg.eigvals(ctds_res['params'].dynamics.weights))))
        lds_m['spec_radius_A'] = float(jnp.max(jnp.abs(
            jnp.linalg.eigvals(lds_res['params'].dynamics.weights))))
        
        sweep_sigQ_results[(sig_q, seed_idx, 'ctds')] = ctds_m
        sweep_sigQ_results[(sig_q, seed_idx, 'lds')] = lds_m

RESULTS['sigQ_sweep'] = sweep_sigQ_results


In [ ]:
# 8.2 — Figure: σ_Q Sweep (Supplementary, 1×4 panels)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sigQ_metrics = [
    ('held_out_ll', 'Held-out LL'),
    ('latent_r2', 'Mean Latent R²'),
    ('E_col_sign_acc', 'E-col Sign Accuracy'),
    ('spec_radius_A', 'Spectral Radius of A_rec'),
]

for pidx, (mkey, ylabel) in enumerate(sigQ_metrics):
    ax = axes[pidx]
    
    for model_name, color, ls in [
        ('ctds', COLORS['ctds'], '-'), ('lds', COLORS['lds'], '--')
    ]:
        means, stds = [], []
        for sig_q in SIGMA_Q_SWEEP:
            vals = [sweep_sigQ_results[(sig_q, s, model_name)][mkey]
                    for s in range(N_DATASET_SEEDS_SWEEP)]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        
        ax.plot(SIGMA_Q_SWEEP, means, color=color, linestyle=ls,
                marker='o', markersize=4, label=model_name.upper())
        ax.fill_between(SIGMA_Q_SWEEP, means - stds, means + stds,
                         color=color, alpha=0.15)
    
    ax.set_xscale('log')
    ax.set_xlabel('σ_Q')
    ax.set_ylabel(ylabel)
    ax.axvline(0.01, color='gray', linestyle=':', alpha=0.5)
    ax.legend(fontsize=7)

fig.suptitle('Recovery vs. Dynamics Noise σ_Q', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_sigQ_sweep.pdf'))
plt.show()


 ## Section 9: Sweep — Latent Dimension Misspecification

In [ ]:
sweep_D_results = {}
fixed_key = DATASET_SEEDS[0]
k_params, k_data, k_split, k_fit = jr.split(fixed_key, 4)

# Generate one fixed dataset with true D=6
true_params_9, ctds_model_9, metadata_9 = make_experiment_params(
    k_params, sigma_q=0.01, sigma_r=0.1
)
y_all_9, x_all_9 = generate_dataset(k_data, true_params_9, ctds_model_9,
                                      B_TRAIN + B_TEST, T)
y_train_9, x_train_9, y_test_9, x_test_9 = train_test_split(
    y_all_9, x_all_9, B_TRAIN, B_TEST, k_split
)

for D_fit in D_SWEEP:
    print(f"\n--- Dimension sweep: D_fit = {D_fit} ---")
    
    D_E_fit = D_fit // 2
    D_I_fit = D_fit - D_E_fit
    
    # Build CTDS model with D_fit dimensions
    cell_types = metadata_9['cell_types']
    cell_sign = metadata_9['cell_sign']
    ctd_fit = jnp.array([D_E_fit, D_I_fit])
    
    ctds_model_fit = CTDS(
        emission_dim=N,
        cell_types=cell_types,
        cell_sign=cell_sign,
        cell_type_dimensions=ctd_fit,
        cell_type_mask=metadata_9['cell_type_mask'],
        region_identity=None,
        inputs_dim=None,
        state_dim=D_fit,
    )
    
    meta_fit = dict(metadata_9)
    meta_fit['D_E'] = D_E_fit
    meta_fit['D_I'] = D_I_fit
    meta_fit['cell_type_dimensions'] = ctd_fit
    meta_fit['dynamics_mask'] = jnp.repeat(cell_sign, ctd_fit)
    
    prefix = f"sec9_D{D_fit}"
    k_c, k_l = jr.split(k_fit)
    k_fit, _ = jr.split(k_fit)  # advance key
    
    ctds_res = fit_ctds(y_train_9, ctds_model_fit, meta_fit,
                        N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
    lds_model_fit = LinearGaussianSSM(state_dim=D_fit, emission_dim=N)
    lds_res = fit_lds(y_train_9, D_fit, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
    
    # Held-out LL (always computable)
    ctds_ll = compute_held_out_ll(ctds_res['params'], y_test_9, ctds_model_fit, 'ctds')
    lds_ll = compute_held_out_ll(lds_res['params'], y_test_9, lds_model_fit, 'lds')
    
    # Subspace angle (always computable)
    x_ctds_sm, _ = get_smoothed_latents(ctds_res['params'], y_test_9, 'ctds')
    x_lds_sm, _ = get_smoothed_latents(lds_res['params'], y_test_9, 'lds')
    
    # For subspace angle, need to handle different dimensions
    # Pad or truncate to compare with true D=6
    x_true_for_angle = x_test_9[:, :, :min(D_fit, D)]
    x_ctds_for_angle = x_ctds_sm[:, :, :min(D_fit, D)]
    x_lds_for_angle = x_lds_sm[:, :, :min(D_fit, D)]
    
    ctds_angle = compute_subspace_angle(x_ctds_for_angle, x_true_for_angle)
    lds_angle = compute_subspace_angle(x_lds_for_angle, x_true_for_angle)
    
    # Per-latent R² for over-specified models
    if D_fit >= D:
        # Align and compute R² for first D dimensions
        x_lds_aligned, _ = align_latents_lds(x_lds_sm, x_test_9)
        # For CTDS, compute R² directly on smoothed latents
        # (alignment is harder when D_fit != D_true)
        per_r2_ctds = np.zeros(D_fit)
        per_r2_lds = np.zeros(D_fit)
        for d in range(min(D_fit, D)):
            _, r2_c = compute_latent_r2(
                x_ctds_sm[:, :, d:d+1],
                x_test_9[:, :, d:d+1]
            )
            per_r2_ctds[d] = r2_c[0]
        # Sort by R² to find best-matching dims
        per_r2_ctds = np.sort(per_r2_ctds)[::-1]
    else:
        per_r2_ctds = np.zeros(D_fit)
        per_r2_lds = np.zeros(D_fit)
    
    sweep_D_results[D_fit] = {
        'ctds_ll': ctds_ll, 'lds_ll': lds_ll,
        'ctds_angle': ctds_angle, 'lds_angle': lds_angle,
        'per_r2_ctds': per_r2_ctds,
    }

RESULTS['D_sweep'] = sweep_D_results


In [ ]:
# 9.2 — Figure: Dimension Sweep (Supplementary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: Held-out LL
ax = axes[0]
ctds_lls = [sweep_D_results[d]['ctds_ll'] for d in D_SWEEP]
lds_lls = [sweep_D_results[d]['lds_ll'] for d in D_SWEEP]
ax.plot(D_SWEEP, ctds_lls, 'o-', color=COLORS['ctds'], label='CTDS')
ax.plot(D_SWEEP, lds_lls, 's--', color=COLORS['lds'], label='LDS')
ax.axvline(D, color='gray', linestyle=':', label=f'D_true={D}')
ax.set_xlabel('D_fit')
ax.set_ylabel('Held-out LL')
ax.legend(fontsize=8)
ax.set_title('Log-Likelihood vs. Fitted Dimension')

# Panel 2: Subspace angle
ax = axes[1]
ctds_angles = [sweep_D_results[d]['ctds_angle'] for d in D_SWEEP]
lds_angles = [sweep_D_results[d]['lds_angle'] for d in D_SWEEP]
ax.plot(D_SWEEP, ctds_angles, 'o-', color=COLORS['ctds'], label='CTDS')
ax.plot(D_SWEEP, lds_angles, 's--', color=COLORS['lds'], label='LDS')
ax.axvline(D, color='gray', linestyle=':')
ax.set_xlabel('D_fit')
ax.set_ylabel('Mean Subspace Angle (degrees)')
ax.legend(fontsize=8)
ax.set_title('Subspace Recovery vs. Fitted Dimension')

# Panel 3: Per-latent R² bar chart
ax = axes[2]
bar_width = 0.2
for i, d_fit in enumerate(D_SWEEP):
    r2s = sweep_D_results[d_fit]['per_r2_ctds']
    x_pos = np.arange(len(r2s)) + i * (len(D_SWEEP) + 1) * bar_width
    colors = ['#2ca02c' if v > 0.1 else '#cccccc' for v in r2s]
    ax.bar(x_pos, r2s, width=bar_width, color=colors, edgecolor='k', linewidth=0.3)
    ax.text(np.mean(x_pos), -0.08, f'D={d_fit}', ha='center', fontsize=8)

ax.axhline(0.1, color='gray', linestyle='--', alpha=0.5, label='R²=0.1 threshold')
ax.set_ylabel('Per-latent R² (CTDS)')
ax.set_title('Latent Recovery by Dimension')
ax.legend(fontsize=7)

fig.suptitle('Latent Dimension Misspecification', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_D_sweep.pdf'))
plt.show()


 ## Section 10: Sweep — Cell-Type Structure Strength (The Money Figure)

In [ ]:
sweep_strength_results = {}

for strength in STRUCTURE_STRENGTH_SWEEP:
    print(f"\n--- Structure strength: s = {strength} ---")
    
    for seed_idx in range(N_DATASET_SEEDS_SWEEP):
        key = DATASET_SEEDS[seed_idx]
        k_params, k_data, k_split, k_fit = jr.split(key, 4)
        
        true_params, ctds_model, metadata = make_experiment_params(
            k_params, sigma_q=0.01, sigma_r=0.1,
            structure_strength=strength
        )
        
        y_all, x_all = generate_dataset(k_data, true_params, ctds_model,
                                         B_TRAIN + B_TEST, T)
        y_train, x_train, y_test, x_test = train_test_split(
            y_all, x_all, B_TRAIN, B_TEST, k_split
        )
        
        prefix = f"sec10_s{strength}_seed{seed_idx}"
        k_c, k_l = jr.split(k_fit)
        
        ctds_res = fit_ctds(y_train, ctds_model, metadata,
                            N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_c, prefix)
        lds_model = LinearGaussianSSM(state_dim=D, emission_dim=N)
        lds_res = fit_lds(y_train, D, N_SEEDS_SWEEP, N_EM_ITERS_SWEEP, k_l, prefix)
        
        ctds_m = compute_all_metrics(ctds_res, true_params, y_test, x_test,
                                     ctds_model, 'ctds', metadata)
        lds_m = compute_all_metrics(lds_res, true_params, y_test, x_test,
                                    lds_model, 'lds', metadata)
        
        sweep_strength_results[(strength, seed_idx, 'ctds')] = ctds_m
        sweep_strength_results[(strength, seed_idx, 'lds')] = lds_m

RESULTS['strength_sweep'] = sweep_strength_results


In [ ]:
# 10.2 — Figure: CTDS Advantage vs. Structure Strength (Thesis Figure 4.6)

# === Main thesis figure: single panel ===
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

advantage_metrics = [
    ('held_out_ll', 'LL Advantage (normalized)', True),
    ('latent_r2', 'Latent R² Advantage', False),
    ('E_col_sign_acc', 'Sign Acc Advantage', False),
]
adv_colors = ['#1b9e77', '#d95f02', '#7570b3']

for midx, (mkey, label, normalize) in enumerate(advantage_metrics):
    means_adv, stds_adv = [], []
    
    for strength in STRUCTURE_STRENGTH_SWEEP:
        ctds_vals = [sweep_strength_results[(strength, s, 'ctds')][mkey]
                     for s in range(N_DATASET_SEEDS_SWEEP)]
        lds_vals = [sweep_strength_results[(strength, s, 'lds')][mkey]
                    for s in range(N_DATASET_SEEDS_SWEEP)]
        
        advantages = [c - l for c, l in zip(ctds_vals, lds_vals)]
        
        if normalize:
            # Normalize by |LDS value|
            norm_advs = [a / (abs(l) + 1e-12) for a, l in zip(advantages, lds_vals)]
            means_adv.append(np.mean(norm_advs))
            stds_adv.append(np.std(norm_advs))
        else:
            means_adv.append(np.mean(advantages))
            stds_adv.append(np.std(advantages))
    
    means_adv, stds_adv = np.array(means_adv), np.array(stds_adv)
    
    ax.plot(STRUCTURE_STRENGTH_SWEEP, means_adv, 'o-',
            color=adv_colors[midx], linewidth=2, label=label)
    ax.fill_between(STRUCTURE_STRENGTH_SWEEP,
                     means_adv - stds_adv, means_adv + stds_adv,
                     color=adv_colors[midx], alpha=0.15)

ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Structure Strength (0 = unstructured → 1 = fully structured)', fontsize=10)
ax.set_ylabel('CTDS Advantage over LDS', fontsize=10)
ax.set_title('CTDS Advantage Grows with True Structure', fontsize=12)
ax.legend(fontsize=9)

fig.savefig(os.path.join(FIG_DIR, 'fig4_6_structure_strength.pdf'))
plt.show()

# === Supplementary: 3-panel version ===
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

supp_metrics = [
    ('held_out_ll', 'CTDS − LDS (Held-out LL)'),
    ('latent_r2', 'CTDS − LDS (Latent R²)'),
    ('E_col_sign_acc', 'E-col Sign Accuracy'),
]

for pidx, (mkey, ylabel) in enumerate(supp_metrics):
    ax = axes[pidx]
    
    if pidx < 2:
        # Plot advantage
        means_adv = []
        for strength in STRUCTURE_STRENGTH_SWEEP:
            ctds_vals = [sweep_strength_results[(strength, s, 'ctds')][mkey]
                         for s in range(N_DATASET_SEEDS_SWEEP)]
            lds_vals = [sweep_strength_results[(strength, s, 'lds')][mkey]
                        for s in range(N_DATASET_SEEDS_SWEEP)]
            means_adv.append(np.mean([c - l for c, l in zip(ctds_vals, lds_vals)]))
        ax.plot(STRUCTURE_STRENGTH_SWEEP, means_adv, 'o-', color=COLORS['ctds'])
        ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
    else:
        # Plot both lines
        for model_name, color, ls in [
            ('ctds', COLORS['ctds'], '-'), ('lds', COLORS['lds'], '--')
        ]:
            means = [np.mean([sweep_strength_results[(s_val, s, model_name)][mkey]
                              for s in range(N_DATASET_SEEDS_SWEEP)])
                     for s_val in STRUCTURE_STRENGTH_SWEEP]
            ax.plot(STRUCTURE_STRENGTH_SWEEP, means, color=color,
                    linestyle=ls, marker='o', label=model_name.upper())
        ax.legend(fontsize=8)
    
    ax.set_xlabel('Structure Strength')
    ax.set_ylabel(ylabel)

fig.suptitle('Structure Strength Sweep (Supplementary)', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_supp_structure_strength.pdf'))
plt.show()


 ## Section 11: Structural Recovery Bar Chart (Thesis Figure 4.3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

bar_metrics = [
    ('E_col_sign_acc', 'E-col\nSign Acc', 'higher'),
    ('I_col_sign_acc', 'I-col\nSign Acc', 'higher'),
    ('zero_block_leakage', 'Zero-block\nLeakage', 'lower'),
    ('nonneg_accuracy', 'Nonneg\nAccuracy', 'higher'),
    ('err_A', 'err_A\n(rel. Frob)', 'lower'),
]

model_order = ['ctds', 'lds', 'scrambled']
model_labels = {'ctds': 'CTDS', 'lds': 'LDS', 'scrambled': 'Scrambled'}
model_colors = [COLORS[m] for m in model_order]

n_groups = len(bar_metrics)
n_bars = len(model_order)
bar_width = 0.22
x_base = np.arange(n_groups)

for bar_idx, model_name in enumerate(model_order):
    means, stds = [], []
    for mkey, _, _ in bar_metrics:
        vals = [base_results[(s, model_name)]['metrics'][mkey]
                for s in range(N_DATASET_SEEDS_BASE)]
        means.append(np.mean(vals))
        stds.append(np.std(vals))
    
    x_pos = x_base + bar_idx * bar_width
    ax.bar(x_pos, means, bar_width, yerr=stds,
           color=model_colors[bar_idx], edgecolor='black', linewidth=0.5,
           label=model_labels[model_name], capsize=3)

# Reference lines
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)

ax.set_xticks(x_base + bar_width * (n_bars - 1) / 2)
ax.set_xticklabels([label for _, label, _ in bar_metrics])
ax.set_ylabel('Metric Value')
ax.set_title('Structural Recovery: CTDS vs. LDS vs. Scrambled', fontsize=12)
ax.legend(loc='upper right')

# Add direction arrows
for i, (_, _, direction) in enumerate(bar_metrics):
    arrow = '↑' if direction == 'higher' else '↓'
    ax.text(x_base[i] + bar_width, ax.get_ylim()[1] * 0.95,
            f'{arrow} better', ha='center', fontsize=7, color='gray')

fig.savefig(os.path.join(FIG_DIR, 'fig4_3_structural_bar_chart.pdf'))
plt.tight_layout()
plt.show()


 ## Section 12: Unified Results Summary

In [ ]:
# 12.1 — Aggregate summary table

print("=" * 80)
print("UNIFIED RESULTS SUMMARY")
print("=" * 80)

# Base experiment summary
print("\n--- Base Experiment ---")
print(summary_df.to_string())

# Sweep summaries
print("\n--- Trial Sweep (CTDS Latent R²) ---")
for B_val in TRIAL_SWEEP:
    vals = [sweep_trial_results[(B_val, s, 'ctds')]['latent_r2']
            for s in range(N_DATASET_SEEDS_SWEEP)]
    print(f"  B={B_val}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")

print("\n--- Structure Strength Sweep (CTDS − LDS Sign Acc) ---")
for s_val in STRUCTURE_STRENGTH_SWEEP:
    ctds_v = [sweep_strength_results[(s_val, s, 'ctds')]['E_col_sign_acc']
              for s in range(N_DATASET_SEEDS_SWEEP)]
    lds_v = [sweep_strength_results[(s_val, s, 'lds')]['E_col_sign_acc']
             for s in range(N_DATASET_SEEDS_SWEEP)]
    print(f"  s={s_val}: CTDS={np.mean(ctds_v):.3f}, "
          f"LDS={np.mean(lds_v):.3f}, "
          f"gap={np.mean(ctds_v) - np.mean(lds_v):.3f}")


 ### 12.2 Six Success Criteria



 1. **Does CTDS fit CTDS-generated data at least as well as LDS on held-out LL?**

    → [Fill in from base experiment table]



 2. **Does CTDS recover latent trajectories better?**

    → [Fill in from latent R² comparison]



 3. **Does CTDS recover sign and block structure substantially better?**

    → [Fill in from structural bar chart]



 4. **Does recovery improve with more data and worsen with more noise?**

    → [Fill in from sweep figures]



 5. **Does the CTDS advantage shrink when cell-type structure is weakened?**

    → [Fill in from structure strength sweep]



 6. **Do the visual heatmaps look like the true matrices?**

    → [Fill in from parameter heatmap figure]

In [ ]:
print("\nNotebook complete. All figures saved to:", FIG_DIR)
print("All cached results in:", CACHE_DIR)
print(f"Total experiments tracked: {len(RESULTS)} sections")
